In [1]:
!pip install -q diffusers transformers accelerate torch gradio opencv-python

In [1]:
import os
import torch
import gradio as gr
from PIL import Image
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video

print("Cargando el modelo de vídeo Stable Video Diffusion (esto puede tardar unos 2 minutos)...")

# 1. Cargar el pipeline de SVD
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16"
)

# Optimización crítica de memoria para evitar Out Of Memory en Colab
pipe.enable_model_cpu_offload()

print("¡Modelo cargado y listo!")

# 2. Función generadora
def animar_imagen(imagen_pil, motion_bucket_id, fps):
    if imagen_pil is None:
        return None

    # SVD trabaja de forma nativa a 1024x576 (proporción 16:9)
    imagen_redimensionada = imagen_pil.resize((1024, 576))

    # Generar los fotogramas del vídeo (25 frames)
    frames = pipe(
        imagen_redimensionada,
        decode_chunk_size=4,       # Procesa de 4 en 4 fotogramas para no colapsar la RAM
        motion_bucket_id=motion_bucket_id,  # Cantidad de movimiento
        noise_aug_strength=0.1
    ).frames[0]

    # Guardar en archivo temporal
    output_path = "video_generado.mp4"
    export_to_video(frames, output_path, fps=fps)

    return output_path

# 3. Interfaz visual interactiva
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 Generador de Vídeo con IA (Stable Video Diffusion)")
    gr.Markdown("Sube una imagen para transformarla en un clip de vídeo con movimiento cinemático.")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="Imagen de origen")

            motion_slider = gr.Slider(
                minimum=1, maximum=255, value=127, step=1,
                label="Intensidad de movimiento (Motion Bucket)",
                info="Valores bajos = movimiento suave y sutil. Valores altos = más acción y dinamismo."
            )
            fps_slider = gr.Slider(
                minimum=5, maximum=15, value=7, step=1,
                label="Fotogramas por segundo (FPS)",
                info="A 7 FPS produce un clip de ~3.5 segundos con velocidad natural."
            )
            btn_generar = gr.Button("Generar Vídeo", variant="primary")

        with gr.Column():
            output_video = gr.Video(label="Vídeo resultante (.mp4)")

    btn_generar.click(
        fn=animar_imagen,
        inputs=[input_img, motion_slider, fps_slider],
        outputs=output_video
    )

demo.launch(share=True)

Cargando el modelo de vídeo Stable Video Diffusion (esto puede tardar unos 2 minutos)...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/496 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

¡Modelo cargado y listo!


/tmp/ipykernel_1037/1103576129.py:45: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de5a4168e504c70061.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
